### Aravec Embeddings

Loading Aravect

In [87]:
import re
import math
import random
import numpy as np
import pandas as pd
from collections import Counter
from typing import List, Dict, Tuple, Optional
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
import gensim

In [88]:
import pathlib as path
import gensim

model_path = path.Path(r"C:\\ghoufrane\\Master\\M2\\llms\\Hanwritten-code-For-LLM-FineTuning\\models\\AraVec\\tweets_sg_100")
aravec_model = gensim.models.Word2Vec.load(str(model_path))

train = pd.read_csv(r'..\\Data\\cleaned_splits\\ARAVEC\\train-ARAVEC.csv')
val = pd.read_csv(r'..\\Data\\cleaned_splits\\ARAVEC\\train-ARAVEC.csv')

In [89]:
url_test = "https://raw.githubusercontent.com/iabufarha/ArSarcasm-v2/main/ArSarcasm-v2/testing_data.csv"
test = pd.read_csv(url_test)
print(f"Test size:  {len(test)}")

Test size:  3000


In [90]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        
set_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilisation du device : {DEVICE}")

Utilisation du device : cuda


Le taux d'OOV (Out-Of-Vocabulary Rate) est le pourcentage de mots inconnus présents dans un nouvel échantillon de texte (ensemble de test) par rapport au vocabulaire de référence utilisé. 

### Tokenization

In [91]:
def tokenize(text):
    if not isinstance(text,str):
        return []
    return text.strip().split()

train['tokens'] = train['tweet'].apply(tokenize)
val['tokens'] = val['tweet'].apply(tokenize)
test['tokens'] = test['tweet'].apply(tokenize)

train[['tweet', 'tokens']].head(3)

,tweet,tokens
0,الغشيم فعلا يا هيموتك يا هيزنقك تموته اخطر خصم...,"[الغشيم, فعلا, يا, هيموتك, يا, هيزنقك, تموته, ..."
1,نشط الان في لبنان روحاني لافروف,"[نشط, الان, في, لبنان, روحاني, لافروف]"
2,كن شريفا امينا لا لان الناس يستحقون الشرف والا...,"[كن, شريفا, امينا, لا, لان, الناس, يستحقون, ال..."


In [92]:
all_train_words = [word for tokens in train['tokens'] for word in tokens]
word_counts = Counter(all_train_words)

word2idx = {
    '<PAD>': 0,
    '<UNK>': 1
}

MIN_FREQ = 2
for word, count in word_counts.items():
    if count >= MIN_FREQ:
        word2idx[word] = len(word2idx)

idx2word = {idx: word for word, idx in word2idx.items()}


print(f"Nombre total de mots uniques dans Train  : {len(word_counts):,}")
print(f"Mots éliminés (fréquence < {MIN_FREQ})        : {sum(1 for c in word_counts.values() if c < MIN_FREQ):,}")
print(f"Mots conservés (fréquence >= {MIN_FREQ})      : {len(word2idx) - 2:,}")
print(f"Taille finale du vocabulaire (+PAD,+UNK) : {len(word2idx):,}")

Nombre total de mots uniques dans Train  : 40,260
Mots éliminés (fréquence < 2)        : 26,227
Mots conservés (fréquence >= 2)      : 14,033
Taille finale du vocabulaire (+PAD,+UNK) : 14,035


### Calcule d'OOV

Token-level OOV = "Quel pourcentage des mots du texte brut sont remplacés par <UNK> ?"

Type-level OOV = "Quel pourcentage des mots du dictionnaire sont inconnus ?"

In [93]:
def get_oov_rates(tokens_series, vocab_dict, split_name="Validation"):
    all_tokens = [w for tokens in tokens_series for w in tokens]
    total_tokens = len(all_tokens)

    oov_tokens = [w for w in all_tokens if w not in vocab_dict or vocab_dict[w] == 1]

    unique_words = set(all_tokens)
    unique_oov_words = set(oov_tokens)

    token_oov_rate = (len(oov_tokens) / total_tokens) * 100 if total_tokens > 0 else 0
    type_oov_rate = (len(unique_oov_words) / len(unique_words)) * 100 if len(unique_words) > 0 else 0

    print(f"--- Taux d'OOV pour {split_name} ---")
    print(f"• Token-level OOV : {token_oov_rate:.2f}% ({len(oov_tokens):,} / {total_tokens:,} mots)")
    print(f"• Type-level OOV  : {type_oov_rate:.2f}% ({len(unique_oov_words):,} / {len(unique_words):,} mots uniques)")
    
    return token_oov_rate, type_oov_rate


val_token_oov, val_type_oov = get_oov_rates(val['tokens'], word2idx, split_name="Validation")

if 'test' in globals() and test is not None:
    test_token_oov, test_type_oov = get_oov_rates(test['tokens'], word2idx, split_name="Test")


--- Taux d'OOV pour Validation ---
• Token-level OOV : 16.56% (26,227 / 158,336 mots)
• Type-level OOV  : 65.14% (26,227 / 40,260 mots uniques)
--- Taux d'OOV pour Test ---
• Token-level OOV : 55.52% (28,068 / 50,556 mots)
• Type-level OOV  : 79.37% (19,617 / 24,715 mots uniques)


In [94]:
def words_to_ids(tokens):
    return [word2idx.get(w, 1) for w in tokens] # 1 est l'index de <UNK>

train['token_ids'] = train['tokens'].apply(words_to_ids)
val['token_ids'] = val['tokens'].apply(words_to_ids)

if 'test' in globals() and test is not None:
    test['token_ids'] = test['tokens'].apply(words_to_ids)

train[['tokens', 'token_ids']].head(3)


,tokens,token_ids
0,"[الغشيم, فعلا, يا, هيموتك, يا, هيزنقك, تموته, ...","[1, 2, 3, 1, 3, 1, 1, 4, 5, 6, 7]"
1,"[نشط, الان, في, لبنان, روحاني, لافروف]","[8, 9, 6, 10, 11, 12]"
2,"[كن, شريفا, امينا, لا, لان, الناس, يستحقون, ال...","[13, 1, 1, 14, 15, 16, 17, 18, 1, 19, 20, 21, ..."


### clean_str

In [95]:
import re

def clean_str_aravec(text):
    """
    Fonction officielle AraVec de normalisation de texte arabe
    à appliquer avant la recherche dans aravec_model.wv.
    """
    if not isinstance(text, str):
        return ""
        
    search = ["أ","إ","ء","آ","ة","_","-","/",".","،"," و "," يا ",'"',"ـ","'","ى","\\",'\n', '\t','&quot;','?','؟','!']
    replace = ["ا","ا","ا","ه"," "," ","","",""," و"," يا","","","","ي","",' ', ' ',' ',' ? ',' ؟ ',' ! ']
    
    # 1. Suppression du tashkeel (diacritiques)
    p_tashkeel = re.compile(r'[\u0617-\u061A\u064B-\u0652]')
    text = re.sub(p_tashkeel, "", text)
    
    # 2. Réduction des allongements de lettres
    p_longation = re.compile(r'(.)\1+')
    subst = r"\1\1"
    text = re.sub(p_longation, subst, text)
    
    text = text.replace('وو', 'و')
    text = text.replace('يي', 'ي')
    text = text.replace('اا', 'ا')
    
    # 3. Remplacement des caractères
    for s, r in zip(search, replace):
        text = text.replace(s, r)
        
    return text.strip()


### Croisement de notre vocabulaire avec AraVec

In [96]:
vocab_size = len(word2idx)
embed_dim = aravec_model.wv.vector_size  # 100 dimensions
print(f"Vocabulaire : {vocab_size} mots | Dimension des vecteurs AraVec : {embed_dim}")

Vocabulaire : 14035 mots | Dimension des vecteurs AraVec : 100


In [97]:
weights_matrix = np.random.normal(scale=0.02, size=(vocab_size, embed_dim)).astype(np.float32)
weights_matrix[0] = np.zeros(embed_dim, dtype=np.float32)


In [98]:
matched = 0
unmatched = 0
for word, idx in word2idx.items():
    if idx == 0:  # Ignorer <PAD>
        continue
    
    cleaned_word = clean_str_aravec(word)
    
    if cleaned_word in aravec_model.wv:
        weights_matrix[idx] = aravec_model.wv[cleaned_word]
        matched += 1
    elif word in aravec_model.wv:
        # Essai avec le mot non nettoyé au cas où
        weights_matrix[idx] = aravec_model.wv[word]
        matched += 1
    else:
        # Mot absent d'AraVec : il conserve son vecteur aléatoire
        unmatched += 1
coverage = (matched / (vocab_size - 1)) * 100
print(f"\n--- Bilan de la recherche dans AraVec ---")
print(f"• Mots trouvés dans AraVec   : {matched:,} ({coverage:.2f}%)")
print(f"• Mots absents (init hasard) : {unmatched:,} ({100 - coverage:.2f}%)")



--- Bilan de la recherche dans AraVec ---
• Mots trouvés dans AraVec   : 13,495 (96.16%)
• Mots absents (init hasard) : 539 (3.84%)


### les mots manquants

In [99]:
missing_words = [word for word, idx in word2idx.items() if idx != 0 and clean_str_aravec(word) not in aravec_model.wv and word not in aravec_model.wv]
print(f"Total mots manquants : {len(missing_words)}")
print("Exemples de mots manquants :", missing_words[:25])


Total mots manquants : 539
Exemples de mots manquants : ['<UNK>', 'لانك', 'ابنك', 'حزب', 'مصر', 'المصيبه', 'انك', '9', 'هاجمت', 'السينماي', 'لجدل', 'فوغـا', 'كـلوسيت', 'مـاكس', 'مامـاز', 'بابـاز', 'خاشفجي', 'عنكم', '10', 'بالعكس', 'المصرين', 'افتاح', '3', 'المصري', '19']


In [100]:
weights_tensor = torch.tensor(weights_matrix, dtype=torch.float32)

# 1. MATRICES X ET Y AVEC PADDING FIXE


In [101]:

MAX_LEN = 50  # 99% des tweets arabes ont moins de 40 mots, 50 est optimal

def pad_sequence(seq, max_len=MAX_LEN):
    """Tronque ou comble avec 0 (<PAD>) jusqu'à max_len."""
    if len(seq) >= max_len:
        return seq[:max_len]
    return seq + [0] * (max_len - len(seq))

# --- Entrées X (Identifiants de tokens) ---
X_train = torch.tensor([pad_sequence(s) for s in train['token_ids']], dtype=torch.long)
X_val   = torch.tensor([pad_sequence(s) for s in val['token_ids']],   dtype=torch.long)

# --- Sorties y (Labels 0.0 ou 1.0) ---
y_train = torch.tensor(train['sarcasm'].astype(float).values, dtype=torch.float32)
y_val   = torch.tensor(val['sarcasm'].astype(float).values,   dtype=torch.float32)

# Préparation du test si la colonne sarcasm est présente
if 'test' in globals() and 'token_ids' in test.columns and 'sarcasm' in test.columns:
    X_test = torch.tensor([pad_sequence(s) for s in test['token_ids']], dtype=torch.long)
    y_test = torch.tensor(test['sarcasm'].astype(float).values, dtype=torch.float32)
else:
    X_test, y_test = None, None

# Vérification immédiate des dimensions
print("=" * 50)
print(f"✅ X_train : {X_train.shape} | y_train : {y_train.shape}")
print(f"✅ X_val   : {X_val.shape}   | y_val   : {y_val.shape}")
if X_test is not None:
    print(f"✅ X_test  : {X_test.shape}  | y_test  : {y_test.shape}")
print("=" * 50)


✅ X_train : torch.Size([10444, 50]) | y_train : torch.Size([10444])
✅ X_val   : torch.Size([10444, 50])   | y_val   : torch.Size([10444])
✅ X_test  : torch.Size([3000, 50])  | y_test  : torch.Size([3000])


# 2. FABRIQUE D'EMBEDDING PROPRE

In [102]:
def get_embedding(mode="unfrozen"):
    """
    Génère une couche d'embedding neuve pour éviter toute contamination entre expériences.
    Modes disponibles : 'random', 'frozen', 'unfrozen'
    """
    if mode == "frozen":
        return nn.Embedding.from_pretrained(
            weights_tensor.clone(),
            freeze=True,
            padding_idx=0
        )
    elif mode == "unfrozen":
        return nn.Embedding.from_pretrained(
            weights_tensor.clone(),
            freeze=False,
            padding_idx=0
        )
    elif mode == "random":
        emb = nn.Embedding(
            num_embeddings=vocab_size, 
            embedding_dim=embed_dim, 
            padding_idx=0
        )
        nn.init.normal_(emb.weight, mean=0.0, std=0.1)
        emb.weight.data[0].fill_(0.0)
        return emb
    else:
        raise ValueError(f"Mode inconnu: {mode}")

# 3. MODÈLE BiLSTM AVEC DROPOUT À 0.5

In [103]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, embedding_layer, hidden_dim=64, dropout=0.5):
        super().__init__()
        self.embedding = embedding_layer
        embed_dim = self.embedding.embedding_dim
        
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )
        # Taux de Dropout fixé à 0.5
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, 1)
    def forward(self, input_ids):
        # 1. Embedding
        x = self.embedding(input_ids)
        
        # 2. LSTM bidirectionnel
        lstm_out, _ = self.lstm(x)
        
        # 3. Max-pooling temporel (ignore les tokens de padding 0)
        pad_mask = (input_ids == 0).unsqueeze(-1)
        lstm_out_masked = lstm_out.masked_fill(pad_mask, -1e9)
        pooled, _ = torch.max(lstm_out_masked, dim=1)
        
        # 4. Couche Dropout 0.5 appliquée avant le classifieur
        out = self.dropout(pooled)
        logits = self.classifier(out).squeeze(-1)
        return logits

In [104]:
class AttentionPooling(nn.Module):
    def __init__(self, input_dim, attention_dim=64):
        super().__init__()
        self.W = nn.Linear(input_dim, attention_dim)
        self.v = nn.Linear(attention_dim, 1, bias=False)

    def forward(self, h, pad_mask=None):
        u = torch.tanh(self.W(h))
        scores = self.v(u).squeeze(-1)
        if pad_mask is not None:
            scores = scores.masked_fill(pad_mask, -1e9)
        alphas = F.softmax(scores, dim=-1)
        c = torch.bmm(alphas.unsqueeze(1), h).squeeze(1)
        return c, alphas


class BiLSTMAttentionClassifier(nn.Module):
    def __init__(self, embedding_layer, hidden_dim=64, attention_dim=64, dropout=0.5):
        super().__init__()
        self.embedding = embedding_layer
        embed_dim = self.embedding.embedding_dim
        
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )
        self.attention = AttentionPooling(input_dim=hidden_dim * 2, attention_dim=attention_dim)
        self.dropout = nn.Dropout(dropout) # Dropout à 0.5
        self.classifier = nn.Linear(hidden_dim * 2, 1)

    def forward(self, input_ids):
        x = self.embedding(input_ids)
        lstm_out, _ = self.lstm(x)
        
        pad_mask = (input_ids == 0)
        c, alphas = self.attention(lstm_out, pad_mask=pad_mask)
        
        out = self.dropout(c) # Dropout 0.5
        logits = self.classifier(out).squeeze(-1)
        return logits


In [105]:
import math

class SinusoidalPositionalEncoding(nn.Module):
    """
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    def __init__(self, d_model=128, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # register_buffer permet de déplacer pe sur GPU automatiquement avec le modèle
        self.register_buffer('pe', pe.unsqueeze(0)) # Taille : (1, max_len, d_model)

    def forward(self, x):
        # x : (batch_size, seq_len, d_model)
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len]


# ==============================================================================
# C3.2. MINI-ENCODEUR TRANSFORMER (2 couches, d_model=128, 4 têtes, Dropout=0.5)
# ==============================================================================

class MiniTransformerClassifier(nn.Module):
    def __init__(
        self, 
        embedding_layer, 
        d_model=128, 
        nhead=4, 
        num_layers=2, 
        use_pe=True,       # Paramètre pour l'ablation (True = Avec PE, False = Sans PE)
        dropout=0.5
    ):
        super().__init__()
        self.use_pe = use_pe
        self.embedding = embedding_layer
        embed_dim = self.embedding.embedding_dim  # 100 (AraVec)
        
        # 1. Projection linéaire : AraVec (100) -> d_model (128)
        self.input_proj = nn.Linear(embed_dim, d_model)
        
        # 2. Encodage positionnel
        self.pos_encoder = SinusoidalPositionalEncoding(d_model=d_model, max_len=MAX_LEN)
        
        # 3. Couches de l'encodeur Transformer (2 couches, 4 têtes)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=256,
            dropout=dropout,          # Dropout=0.5 dans les couches d'attention
            batch_first=True,
            activation="relu"
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # 4. Tête de classification finale avec Dropout 0.5
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, 1)

    def forward(self, input_ids):
        # 1. Embedding + Projection vers d_model (128)
        x = self.embedding(input_ids)       # (batch, seq_len, 100)
        x = self.input_proj(x)              # (batch, seq_len, 128)
        
        # 2. Ajout des positions (Ablation : actif si use_pe=True)
        if self.use_pe:
            x = self.pos_encoder(x)
            
        # 3. Masque de padding (True pour les tokens <PAD> = 0)
        # PyTorch ignore les positions où pad_mask est True dans l'attention
        pad_mask = (input_ids == 0)
        
        # 4. Passage dans les 2 couches Transformer avec masque
        encoded = self.transformer_encoder(x, src_key_padding_mask=pad_mask)
        
        # 5. Masked Average Pooling (moyenne uniquement sur les vrais mots)
        mask_expanded = (~pad_mask).unsqueeze(-1).float() # 1 pour vrais mots, 0 pour PAD
        sum_embeddings = (encoded * mask_expanded).sum(dim=1)
        sum_mask = mask_expanded.sum(dim=1).clamp(min=1e-9)
        pooled = sum_embeddings / sum_mask  # (batch, 128)
        
        # 6. Classification binaire avec Dropout de 0.5
        out = self.dropout(pooled)
        logits = self.classifier(out).squeeze(-1)
        return logits



d'apprentissage des embeddings :

Mode Random (Aléatoire) :

La matrice d'embedding est initialisée aléatoirement (loi normale $\mathcal{N}(0, \sigma^2)$).

Les vecteurs de mots apprennent de zéro (end-to-end) à partir de la tâche de sarcasme via la rétropropagation du gradient (requires_grad = True).

Mode Frozen (Pré-entraîné figé) :

Les vecteurs de mots sont initialisés avec les poids pré-entraînés d'AraVec (Word2Vec entraîné sur des millions de tweets arabes).

La couche d'embedding est verrouillée (freeze=True / requires_grad = False). Seules les couches supérieures (ex. LSTM, couches denses) s'entraînent.

Mode Unfrozen (Pré-entraîné affiné / Fine-tuné) :

Initialisation avec les poids d'AraVec.

Les vecteurs ne sont pas figés (freeze=False). Le modèle part d'une sémantique pré-apprise mais réajuste les représentations pour s'adapter spécifiquement au sarcasme.

### Evaluation metrics

Train Loss & Val Loss (souvent une CrossEntropyLoss ou BCEWithLogitsLoss).

Val F1-sarc : le score F1 spécifique à la classe positive (la classe minoritaire « sarcastique »).

Un mécanisme d'Early Stopping (arrêt précoce) qui interrompt l'entraînement si le Val F1-sarc ne s'améliore plus après un certain nombre d'époques (patience).


# 4. ENTRAÎNEMENT PAR BATCHES

In [106]:
class HybridSarcasmClassifier(nn.Module):
    """
    Réseau profond combinant :
    1. Couche C3 : Mini-encodeur Transformer (2 couches, 4 têtes, d_model=128)
    2. Couche C1 : BiLSTM bidirectionnel (hidden_dim=64 x 2 = 128)
    3. Couche C2 : Pooling Attentionnel implémenté à la main
    4. Régularisation : Dropout = 0.5
    """
    def __init__(
        self,
        embedding_layer,
        d_model=128,
        nhead=4,
        transformer_layers=2,
        lstm_hidden=64,
        attention_dim=64,
        use_pe=True,         # Option : activer ou désactiver l'encodage positionnel
        dropout=0.5          # Dropout fixé à 0.5
    ):
        super().__init__()
        self.use_pe = use_pe
        self.embedding = embedding_layer
        embed_dim = self.embedding.embedding_dim  # 100 dimensions AraVec
        
        # --- A. Entrée vers Transformer ---
        self.input_proj = nn.Linear(embed_dim, d_model)
        self.pos_encoder = SinusoidalPositionalEncoding(d_model=d_model, max_len=MAX_LEN)
        
        # --- B. COUCHE C3 : Mini-Transformer (2 couches, 4 têtes) ---
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=256,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=transformer_layers)
        
        # --- C. COUCHE C1 : BiLSTM ---
        # Prend en entrée la séquence contextualisée par le Transformer (128)
        self.bilstm = nn.LSTM(
            input_size=d_model,
            hidden_size=lstm_hidden,
            batch_first=True,
            bidirectional=True
        )
        
        # --- D. COUCHE C2 : Attention Pooling ---
        # Reçoit la sortie bidirectionnelle du BiLSTM (64 * 2 = 128)
        self.attention = AttentionPooling(input_dim=lstm_hidden * 2, attention_dim=attention_dim)
        
        # --- E. Classifieur final avec Dropout 0.5 ---
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(lstm_hidden * 2, 1)
    def forward(self, input_ids):
        # 1. Embedding + Projection vers 128
        x = self.embedding(input_ids)
        x = self.input_proj(x)
        
        # 2. Couche C3 (Transformer)
        if self.use_pe:
            x = self.pos_encoder(x)
        pad_mask = (input_ids == 0)
        x_trans = self.transformer(x, src_key_padding_mask=pad_mask)
        
        # 3. Couche C1 (BiLSTM)
        lstm_out, _ = self.bilstm(x_trans)
        
        # 4. Couche C2 (Attention Pooling)
        context_vector, alphas = self.attention(lstm_out, pad_mask=pad_mask)
        
        # 5. Dropout 0.5 + Sortie de classification
        out = self.dropout(context_vector)
        logits = self.classifier(out).squeeze(-1)
        return logits


In [107]:
def train_and_validate(model, X_tr, y_tr, X_v, y_v, epochs=15, batch_size=32, lr=1e-3, weight_decay=1e-4, patience=5):
    """
    Entraîne et valide le modèle directement sur les tenseurs en découpant par batch_size.
    - weight_decay=1e-4 : régularisation L2 essentielle pour empêcher le par-cœur
    - Early stopping sur le F1-sarc de validation
    """
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), 
        lr=lr, 
        weight_decay=weight_decay
    )
    criterion = nn.BCEWithLogitsLoss()
    
    n_train = len(X_tr)
    n_val = len(X_v)
    
    best_val_f1 = -1.0
    best_weights = None
    patience_counter = 0
    history = {'train_loss': [], 'val_loss': [], 'val_f1': []}
    for epoch in range(1, epochs + 1):
        # -------------------------------------------------------------
        # 1. ENTRAÎNEMENT (Mélange + mini-batches)
        # -------------------------------------------------------------
        model.train() # Active le Dropout (0.5)
        permutation = torch.randperm(n_train)
        total_train_loss = 0.0
        n_train_batches = 0
        for i in range(0, n_train, batch_size):
            indices = permutation[i:i + batch_size]
            bx = X_tr[indices].to(DEVICE)
            by = y_tr[indices].to(DEVICE)
            optimizer.zero_grad()
            logits = model(bx)
            loss = criterion(logits, by)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
            n_train_batches += 1
        train_loss = total_train_loss / n_train_batches
        # -------------------------------------------------------------
        # 2. VALIDATION (Sur X_val / y_val uniquement)
        # -------------------------------------------------------------
        model.eval() # Désactive le Dropout
        total_val_loss = 0.0
        n_val_batches = 0
        all_preds = []
        all_targets = []
        with torch.no_grad():
            for i in range(0, n_val, batch_size):
                bx = X_v[i:i + batch_size].to(DEVICE)
                by = y_v[i:i + batch_size].to(DEVICE)
                logits = model(bx)
                loss = criterion(logits, by)
                total_val_loss += loss.item()
                n_val_batches += 1
                preds = (torch.sigmoid(logits) >= 0.5).long()
                all_preds.extend(preds.cpu().tolist())
                all_targets.extend(by.cpu().long().tolist())
        val_loss = total_val_loss / n_val_batches
        
        # Métrique officielle du TP : F1 de la classe positive (sarcastique = 1)
        val_f1 = f1_score(all_targets, all_preds, pos_label=1, zero_division=0)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_f1'].append(val_f1)
        print(f"Époque {epoch:02d}/{epochs:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val F1-sarc: {val_f1:.4f}")
        # Early Stopping
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"⏹️ Arrêt précoce (Early Stopping) à l'époque {epoch} (Meilleur Val F1 = {best_val_f1:.4f})")
                break
    if best_weights is not None:
        model.load_state_dict(best_weights)
    return best_val_f1, history


In [108]:
# ==============================================================================
# COMPARAISON DES OPTIONS SUR LE MODÈLE HYBRIDE (C3 + C1 + C2)
# ==============================================================================

options = {
    "1. Modèle Hybride (Random)":            {"mode": "random",   "use_pe": True},
    "2. Modèle Hybride (AraVec Frozen)":     {"mode": "frozen",   "use_pe": True},
    "3. Modèle Hybride (AraVec Unfrozen)":   {"mode": "unfrozen", "use_pe": True},
    "4. Modèle Hybride (Sans Encodage Pos)": {"mode": "unfrozen", "use_pe": False},
}

hybrid_results = {}

for option_name, config in options.items():
    print(f"\n{'='*70}")
    print(f"🚀 ENTRAÎNEMENT : {option_name.upper()}")
    print(f"{'='*70}")
    
    # 1. Générer l'embedding propre selon l'option
    emb = get_embedding(config["mode"])
    
    # 2. Instancier le modèle hybride complet (C3 + C1 + C2) avec Dropout=0.5
    model = HybridSarcasmClassifier(
        embedding_layer=emb,
        d_model=128,
        nhead=4,
        transformer_layers=2,
        lstm_hidden=64,
        attention_dim=64,
        use_pe=config["use_pe"],
        dropout=0.5
    ).to(DEVICE)
    
    # 3. Entraînement et validation par batches (sans DataLoader)
    best_f1, hist = train_and_validate(
        model=model,
        X_tr=X_train,
        y_tr=y_train,
        X_v=X_val,
        y_v=y_val,
        epochs=3,
        batch_size=32,
        lr=1e-3,
        weight_decay=1e-4,
        patience=3
    )
    
    hybrid_results[option_name] = (best_f1, hist)

# -------------------------------------------------------------
# TABLEAU COMPARATIF FINAL DES OPTIONS
# -------------------------------------------------------------
print("\n" + "=" * 65)
print("📊 TABLEAU COMPARATIF DU MODÈLE HYBRIDE (C3 + C1 + C2)")
print("=" * 65)
for option_name, (score, _) in hybrid_results.items():
    print(f"• {option_name:40s} ➔ Val F1-sarc : {score:.4f}")
print("=" * 65)



🚀 ENTRAÎNEMENT : 1. MODÈLE HYBRIDE (RANDOM)


Époque 01/03 | Train Loss: 0.4539 | Val Loss: 0.3863 | Val F1-sarc: 0.0000
Époque 02/03 | Train Loss: 0.3728 | Val Loss: 0.2896 | Val F1-sarc: 0.0629
Époque 03/03 | Train Loss: 0.2769 | Val Loss: 0.1831 | Val F1-sarc: 0.7826

🚀 ENTRAÎNEMENT : 2. MODÈLE HYBRIDE (ARAVEC FROZEN)
Époque 01/03 | Train Loss: 0.4034 | Val Loss: 0.3473 | Val F1-sarc: 0.4676
Époque 02/03 | Train Loss: 0.3565 | Val Loss: 0.3857 | Val F1-sarc: 0.4217
Époque 03/03 | Train Loss: 0.3517 | Val Loss: 0.3410 | Val F1-sarc: 0.5412

🚀 ENTRAÎNEMENT : 3. MODÈLE HYBRIDE (ARAVEC UNFROZEN)
Époque 01/03 | Train Loss: 0.4048 | Val Loss: 0.3062 | Val F1-sarc: 0.4569
Époque 02/03 | Train Loss: 0.3192 | Val Loss: 0.2327 | Val F1-sarc: 0.6308
Époque 03/03 | Train Loss: 0.2411 | Val Loss: 0.1551 | Val F1-sarc: 0.8538

🚀 ENTRAÎNEMENT : 4. MODÈLE HYBRIDE (SANS ENCODAGE POS)
Époque 01/03 | Train Loss: 0.3979 | Val Loss: 0.3296 | Val F1-sarc: 0.5229
Époque 02/03 | Train Loss: 0.3253 | Val Loss: 0.2573 | Val F1-sarc: 0.7426
Époque 03/03 

In [109]:
# ==============================================================================
# 1. PRÉPARATION DES MATRICES DE TEST (X_test, y_test)
# ==============================================================================

# Si les tokens n'ont pas encore été convertis en ids pour le test :
if 'token_ids' not in test.columns:
    if 'tokens' not in test.columns:
        test['tokens'] = test['tweet'].apply(tokenize)
    test['token_ids'] = test['tokens'].apply(words_to_ids)

# 1. Tenseur d'entrées X_test avec padding fixe (MAX_LEN = 50)
X_test = torch.tensor([pad_sequence(s, max_len=MAX_LEN) for s in test['token_ids']], dtype=torch.long)

# 2. Tenseur de cibles y_test (True -> 1.0, False -> 0.0)
y_test = torch.tensor(test['sarcasm'].astype(float).values, dtype=torch.float32)

print("=" * 50)
print(f"✅ X_test prêt : {X_test.shape} (3 000 tweets x {MAX_LEN} tokens)")
print(f"✅ y_test prêt : {y_test.shape} (Distribution : {int((y_test==1).sum())} sarcastiques / {int((y_test==0).sum())} non-sarcastiques)")
print("=" * 50)


✅ X_test prêt : torch.Size([3000, 50]) (3 000 tweets x 50 tokens)
✅ y_test prêt : torch.Size([3000]) (Distribution : 821 sarcastiques / 2179 non-sarcastiques)


In [110]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

# ==============================================================================
# 2. FONCTION D'ÉVALUATION SUR LE JEU DE TEST
# ==============================================================================

def evaluate_on_test(model, X_te, y_te, batch_size=32, threshold=0.5):
    """
    Évalue le modèle sur le jeu de test par mini-batches sans DataLoader.
    Affiche le rapport de classification complet et la matrice de confusion.
    """
    model.eval()
    criterion = nn.BCEWithLogitsLoss()
    
    total_loss = 0.0
    n_batches = 0
    all_preds = []
    all_probs = []
    all_targets = []

    with torch.no_grad():
        for i in range(0, len(X_te), batch_size):
            bx = X_te[i:i + batch_size].to(DEVICE)
            by = y_te[i:i + batch_size].to(DEVICE)

            logits = model(bx)
            loss = criterion(logits, by)
            total_loss += loss.item()
            n_batches += 1

            probs = torch.sigmoid(logits)
            preds = (probs >= threshold).long()

            all_probs.extend(probs.cpu().tolist())
            all_preds.extend(preds.cpu().tolist())
            all_targets.extend(by.cpu().long().tolist())

    test_loss = total_loss / n_batches
    
    # Métrique officielle du TP : F1 sur la classe sarcastique (1)
    f1_sarc = f1_score(all_targets, all_preds, pos_label=1, zero_division=0)
    acc = accuracy_score(all_targets, all_preds)
    macro_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)

    print("=" * 65)
    print("🎯 RÉSULTATS OFFICIELS SUR LE JEU DE TEST (ArSarcasm-v2)")
    print("=" * 65)
    print(f"• Test Loss                : {test_loss:.4f}")
    print(f"• F1-Score Sarcasme (pos=1): {f1_sarc:.4f}  <-- Métrique officielle")
    print(f"• Macro F1-Score           : {macro_f1:.4f}")
    print(f"• Accuracy                 : {acc:.4f} ({acc*100:.2f}%)")
    
    print("\n📊 RAPPORT DE CLASSIFICATION DÉTAILLÉ :")
    print(classification_report(
        all_targets, 
        all_preds, 
        target_names=["Non-sarcastique (0)", "Sarcastique (1)"], 
        digits=4
    ))
    
    cm = confusion_matrix(all_targets, all_preds)
    print("🧩 MATRICE DE CONFUSION :")
    print(f"  Vrais Négatifs (TN)  : {cm[0, 0]:4d} | Faux Positifs (FP) : {cm[0, 1]:4d}")
    print(f"  Faux Négatifs (FN)   : {cm[1, 0]:4d} | Vrais Positifs (TP): {cm[1, 1]:4d}")
    print("=" * 65)
    
    return {
        "loss": test_loss,
        "f1_sarc": f1_sarc,
        "macro_f1": macro_f1,
        "accuracy": acc,
        "cm": cm
    }


In [111]:
# ==============================================================================
# 3. LANCER LE TEST
# ==============================================================================

test_results = evaluate_on_test(model, X_test, y_test, batch_size=32, threshold=0.5)


🎯 RÉSULTATS OFFICIELS SUR LE JEU DE TEST (ArSarcasm-v2)
• Test Loss                : 0.6103
• F1-Score Sarcasme (pos=1): 0.3951  <-- Métrique officielle
• Macro F1-Score           : 0.6074
• Accuracy                 : 0.7223 (72.23%)

📊 RAPPORT DE CLASSIFICATION DÉTAILLÉ :
                     precision    recall  f1-score   support

Non-sarcastique (0)     0.7754    0.8697    0.8198      2179
    Sarcastique (1)     0.4892    0.3313    0.3951       821

           accuracy                         0.7223      3000
          macro avg     0.6323    0.6005    0.6074      3000
       weighted avg     0.6971    0.7223    0.7036      3000

🧩 MATRICE DE CONFUSION :
  Vrais Négatifs (TN)  : 1895 | Faux Positifs (FP) :  284
  Faux Négatifs (FN)   :  549 | Vrais Positifs (TP):  272
